In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('../input/'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### Part I: Data Importation and Exploration

In this part, we will do the following:
* Import data, inpsect data dimensions, dtypes and data summary
* Inspect data missing values proportions 
* Understanding data by visualizations, data metrics and analyses

In [ ]:
# Import data
train_app = pd.read_csv("../input/home-credit-default-risk/application_train.csv")
test_app = pd.read_csv("../input/home-credit-default-risk/application_test.csv")
bureau = pd.read_csv("../input/home-credit-default-risk/bureau.csv")
bureau_bal = pd.read_csv("../input/home-credit-default-risk/bureau_balance.csv")
ccard_bal = pd.read_csv("../input/home-credit-default-risk/credit_card_balance.csv")
install_pay = pd.read_csv("../input/home-credit-default-risk/installments_payments.csv")
pos_bal = pd.read_csv("../input/home-credit-default-risk/POS_CASH_balance.csv")
prev_app = pd.read_csv("../input/home-credit-default-risk/previous_application.csv")

preprocessing main train and test

In [ ]:
def preprocess_main_enhanced(df):
    df['DAYS_EMPLOYED_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']

    df["EXTSOURCE_MEAN"] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)

    df['EXTSOURCES_GM'] = pow(df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3'], 1/3)

    df['ANNUITY_CREDIT_RATIO'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']

    df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

    df['INCOME_CREDIT_RATIO'] = df['AMT_INCOME_TOTAL'] / df['AMT_CREDIT']

    df["CREDIT_GOODS_RATIO"] = df["AMT_CREDIT"]/df["AMT_GOODS_PRICE"]

    df["CREDIT_GOODS_DIFF"] = df["AMT_CREDIT"] - df["AMT_GOODS_PRICE"]
    
 
    df['EXT_SOURCES_WEIGHTED'] = (
        df['EXT_SOURCE_1'] * 0.4 + 
        df['EXT_SOURCE_2'] * 0.3 + 
        df['EXT_SOURCE_3'] * 0.3
    )
    

    df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / df['AMT_CREDIT']
    
    df['AGE_YEARS'] = df['DAYS_BIRTH'] / -365
    df['EMPLOYED_YEARS'] = df['DAYS_EMPLOYED'] / -365
    df['EMPLOYED_BIRTH_RATIO'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH']
    
    df['DOWN_PAYMENT'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT']
    df['DOWN_PAYMENT_RATIO'] = df['DOWN_PAYMENT'] / df['AMT_GOODS_PRICE']
    
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
    df['EXT_SOURCES_MAX'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].max(axis=1)
    df['EXT_SOURCES_MIN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].min(axis=1)
    
    return df

In [ ]:
train_app = preprocess_main_enhanced(train_app)
test_app = preprocess_main_enhanced(test_app)

preprocess installments

In [ ]:
def preprocess_installments(df):
    df['DAYS_DELAY_PAYMENT'] = df['DAYS_ENTRY_PAYMENT']-df['DAYS_INSTALMENT']
    df['OVERDUE_AMT_PAYMENT'] = df['AMT_INSTALMENT']-df['AMT_PAYMENT']
    df['OVERDUE_AMT_PAYMENT_RELATIVE'] = 0
    df.loc[df['AMT_INSTALMENT'] != 0, 'OVERDUE_AMT_PAYMENT_RELATIVE'] = df['AMT_PAYMENT']/df['AMT_INSTALMENT']

    return df

In [ ]:
install_pay = preprocess_installments(install_pay)

preprocess ccard_bal

In [ ]:
ccard_bal['BALANCE_EXCEED_LIMIT'] = ccard_bal['AMT_BALANCE'] - ccard_bal['AMT_CREDIT_LIMIT_ACTUAL']
ccard_bal['REPAYMENT_EXCEED_MININSTAL'] = ccard_bal['AMT_PAYMENT_TOTAL_CURRENT'] - ccard_bal['AMT_INST_MIN_REGULARITY']
ccard_bal['SK_NET_DPD'] = ccard_bal['SK_DPD'] - ccard_bal['SK_DPD_DEF']

preprocessing prev_app

In [ ]:
prev_app['CREDIT_NET_APPLICATION'] = prev_app['AMT_CREDIT'] - prev_app['AMT_APPLICATION'] #difference in credit alloted vs applied for
prev_app['CREDIT_NET_AMT_GOOD_PRICE'] = prev_app['AMT_CREDIT'] - prev_app['AMT_GOODS_PRICE'] #difference in credit alloted vs applied for
prev_app.reset_index(drop=True)
prev_app['AMT_DOWN_PAYMENT'] = prev_app['AMT_DOWN_PAYMENT'].fillna(0)

In [ ]:
## Dimesions of data
print("Train set dimension:", train_app.shape)
print("Test set dimension:", test_app.shape)
print("bureau set dimension:", bureau.shape)
print("bureau_bal set dimension:", bureau_bal.shape)
print("ccard_bal set dimension:", ccard_bal.shape)
print("install_pay set dimension:", install_pay.shape)
print("pos_bal set dimension:", pos_bal.shape)
print("prev_app set dimension:", prev_app.shape)

conduct data aggregation

In [ ]:
def create_temporal_aggregations(df, df_name):
    
    group_col = 'SK_ID_CURR'  # Hardcode since we always group by this
    
    # Select numeric columns for aggregation (excluding group column)
    numeric_cols = df.select_dtypes(include='number').columns
    numeric_cols = numeric_cols[numeric_cols != group_col]
    
    # Basic aggregations
    agg_dict = {
        col: ['mean', 'std', 'min', 'max', 'sum', 'count'] 
        for col in numeric_cols
    }
    
    # Add categorical aggregations if any
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    agg_dict.update({
        col: ['count', 'nunique', 'first', 'last',  'size'] 
        for col in categorical_cols
    })
    
    basic_agg = df.groupby(group_col).agg(agg_dict)
    basic_agg.columns = [f'{col[0]}_{col[1]}_{df_name}' for col in basic_agg.columns]
    
    # Temporal aggregations for previous applications only
    if df_name == 'prev_app' and 'DAYS_DECISION' in df.columns:
        print("Creating temporal aggregations for previous applications...")
        
        # Recent applications (last 30, 90, 180 days)
        for days in [30, 90, 180]:
            recent_mask = df['DAYS_DECISION'] >= -days
            if recent_mask.any():
                print(f"  Last {days} days: {recent_mask.sum()} records")
                
                # Select numeric columns excluding DAYS_DECISION
                temp_numeric_cols = [col for col in numeric_cols if col != 'DAYS_DECISION']
                
                recent_agg = df[recent_mask].groupby(group_col).agg({
                    col: ['mean', 'sum', 'count'] 
                    for col in temp_numeric_cols[:15]  # Limit to avoid memory issues
                })
                recent_agg.columns = [f'{col[0]}_{col[1]}_LAST{days}D_{df_name}' for col in recent_agg.columns]
                basic_agg = basic_agg.merge(recent_agg, on=group_col, how='left')
        
        # First/last application differences
        first_last_diff = df.groupby(group_col).agg({
            'AMT_CREDIT': ['first', 'last'],
            'AMT_ANNUITY': ['first', 'last'],
            'DAYS_DECISION': ['first', 'last']
        })
        first_last_diff.columns = [f'{col[0]}_{col[1]}_{df_name}' for col in first_last_diff.columns]
        
        # Calculate differences
        for col in ['AMT_CREDIT', 'AMT_ANNUITY']:
            basic_agg[f'{col}_FIRST_LAST_DIFF_{df_name}'] = (
                first_last_diff[f'{col}_first_{df_name}'] - first_last_diff[f'{col}_last_{df_name}']
            )
    
    return basic_agg.reset_index()

In [ ]:
def aggregate(df,df_name):
    numeric_cols = df.select_dtypes(include='number').columns
    # df = df.select_dtypes(exclude=['object'])
    object_cols = df.select_dtypes(include='object').columns
    numeric_cols = numeric_cols[numeric_cols != 'SK_ID_PREV' ]
    agg_dict = {}
    agg_dict.update({col: ['sum', 'mean', 'count', 'last','min','max','median','first'] for col in numeric_cols})
    agg_dict.update({col: ['count', 'nunique', 'first', 'last'] for col in object_cols})
    result = df.groupby('SK_ID_CURR').agg(agg_dict)
    result.columns = [f'{col}_{func}_{df_name}' for col, func in result.columns]
    result = result.reset_index()
    return result

In [ ]:
install_pay_after_aggregation = aggregate(install_pay,"install_pay")
ccard_bal_after_aggregation = aggregate(ccard_bal,"ccard_bal")
pos_bal_after_aggregation = aggregate(pos_bal,"pos_bal")
prev_app_after_aggregation = create_temporal_aggregations(prev_app,"prev_app")

bureau_after_aggregation = aggregate(bureau,"bureau")

concatnate train and test data

In [ ]:
external_datas = [install_pay_after_aggregation,ccard_bal_after_aggregation,pos_bal_after_aggregation,prev_app_after_aggregation,bureau_after_aggregation]

train = train_app
test = test_app
for external_data in external_datas:
    train = train.merge(external_data, on='SK_ID_CURR', how='left')
for external_data in external_datas:
    test = test.merge(external_data, on='SK_ID_CURR', how='left')

In [ ]:
def select_features_ridge(X_train, y_train, X_test, n_features=300):
    """Simple feature selection using Ridge regression (Bojan's approach)"""
    from sklearn.linear_model import Ridge
    from sklearn.feature_selection import SelectFromModel
    
    # Use Ridge for feature selection
    ridge = Ridge(alpha=1.0, random_state=42)
    ridge.fit(X_train, y_train)
    
    # Select top features
    selector = SelectFromModel(ridge, max_features=n_features, prefit=True)
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    feature_mask = selector.get_support()
    selected_features = X_train.columns[feature_mask]
    
    print(f"Selected {len(selected_features)} features from {X_train.shape[1]} total")
    
    return X_train_selected, X_test_selected, selected_features

In [ ]:
ID_for_test = test['SK_ID_CURR']

In [ ]:

def reduce_memory_usage(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """
    Reduces memory usage of a pandas DataFrame by converting column types.

    Args:
    - df (pd.DataFrame): DataFrame to optimize.
    - name (str): Name of the DataFrame.

    Returns:
    - pd.DataFrame: Optimized DataFrame.
    """
    print(f"Memory usage of dataframe \"{name}\" is {round(df.memory_usage(deep=True).sum() / 1024**2, 4)} MB.")

    # Create a copy to avoid modifying the original during iteration
    df_optimized = df.copy()
    
    for col in df_optimized.columns:
        col_type = df_optimized[col].dtype
        
        # Handle numeric columns
        if np.issubdtype(col_type, np.integer) or np.issubdtype(col_type, np.floating):
            c_min = df_optimized[col].min()
            c_max = df_optimized[col].max()
            
            if not pd.isna(c_min) and not pd.isna(c_max):
                # Integer types
                if np.issubdtype(col_type, np.integer):
                    if c_min >= 0:
                        if c_max <= np.iinfo(np.uint8).max:
                            df_optimized[col] = df_optimized[col].astype(np.uint8)
                        elif c_max <= np.iinfo(np.uint16).max:
                            df_optimized[col] = df_optimized[col].astype(np.uint16)
                        elif c_max <= np.iinfo(np.uint32).max:
                            df_optimized[col] = df_optimized[col].astype(np.uint32)
                        elif c_max <= np.iinfo(np.uint64).max:
                            df_optimized[col] = df_optimized[col].astype(np.uint64)
                    else:
                        if (c_min >= np.iinfo(np.int8).min and 
                            c_max <= np.iinfo(np.int8).max):
                            df_optimized[col] = df_optimized[col].astype(np.int8)
                        elif (c_min >= np.iinfo(np.int16).min and 
                              c_max <= np.iinfo(np.int16).max):
                            df_optimized[col] = df_optimized[col].astype(np.int16)
                        elif (c_min >= np.iinfo(np.int32).min and 
                              c_max <= np.iinfo(np.int32).max):
                            df_optimized[col] = df_optimized[col].astype(np.int32)
                        elif (c_min >= np.iinfo(np.int64).min and 
                              c_max <= np.iinfo(np.int64).max):
                            df_optimized[col] = df_optimized[col].astype(np.int64)
                
                # Float types
                elif np.issubdtype(col_type, np.floating):
                    if (c_min > np.finfo(np.float32).min and 
                        c_max < np.finfo(np.float32).max):
                        df_optimized[col] = df_optimized[col].astype(np.float32)
    
    print(f"Memory usage of dataframe \"{name}\" became {round(df_optimized.memory_usage(deep=True).sum() / 1024**2, 4)} MB.")
    
    return df_optimized

In [ ]:
train = reduce_memory_usage(train,"train")
test = reduce_memory_usage(test,"test")

conduct training and testing

Deal with categorical features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, KFold
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
import gc



In [ ]:
import re
import pandas as pd

def clean_feature_names(df):
    """Remove special characters from column names for LightGBM compatibility"""
    df_clean = df.copy()
    
    # Clean feature names - remove or replace special characters
    clean_columns = []
    for col in df_clean.columns:
        # Replace special characters with underscore
        clean_col = re.sub(r'[^a-zA-Z0-9_]', '_', str(col))
        # Remove multiple consecutive underscores
        clean_col = re.sub(r'_+', '_', clean_col)
        # Remove leading/trailing underscores
        clean_col = clean_col.strip('_')
        # Ensure it doesn't start with number
        if clean_col and clean_col[0].isdigit():
            clean_col = 'feature_' + clean_col
        # If empty after cleaning, give generic name
        if not clean_col:
            clean_col = 'feature'
        
        clean_columns.append(clean_col)
    
    df_clean.columns = clean_columns
    return df_clean

# Apply to your data BEFORE training
train = clean_feature_names(train)
test = clean_feature_names(test)
# # Now use X_clean in your training
# print("Original columns:", X.columns.tolist()[:10])  # First 10 columns
# print("Cleaned columns:", X_clean.columns.tolist()[:10])

create categorical features with train and test

In [ ]:
from sklearn.preprocessing import LabelEncoder
def smart_categorical_encoding_optimized(train_df, test_df, target_col, low_cardinality_threshold=20):
    """
    Optimized version with better performance and error handling
    """
    
    # Create copies to avoid modifying original data
    X_train = train_df.drop(target_col, axis=1).copy()
    y_train = train_df[target_col].copy()
    X_test = test_df.copy()
    
    # Identify categorical columns
    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Separate low and high cardinality features
    low_cardinality_cols = []
    high_cardinality_cols = []
    
    for col in categorical_cols:
        unique_count = X_train[col].nunique()
        if unique_count <= low_cardinality_threshold:
            low_cardinality_cols.append(col)
        else:
            high_cardinality_cols.append(col)
    
    print(f"Low cardinality features ({len(low_cardinality_cols)}): {low_cardinality_cols}")
    print(f"High cardinality features ({len(high_cardinality_cols)}): {high_cardinality_cols}")
    
    # Process low cardinality features - using manual mapping (more robust)
    for col in low_cardinality_cols:
        # Fill missing values
        X_train[col] = X_train[col].fillna('MISSING')
        X_test[col] = X_test[col].fillna('MISSING')
        
        # Get all unique values from both datasets
        all_unique = sorted(set(X_train[col].unique()) | set(X_test[col].unique()))
        
        # Create mapping dictionary
        value_to_code = {val: idx for idx, val in enumerate(all_unique)}
        
        # Apply mapping (faster than apply)
        X_train[col] = X_train[col].map(value_to_code)
        X_test[col] = X_test[col].map(value_to_code)
        
        # Fill any NaN values (for unseen categories in test set)
        X_train[col] = X_train[col].fillna(-1).astype(int)
        X_test[col] = X_test[col].fillna(-1).astype(int)
    
    # Process high cardinality features - use label encoding
    label_encoders = {}
    for col in high_cardinality_cols:
        le = LabelEncoder()
        
        # Fill missing values
        X_train[col] = X_train[col].fillna('MISSING')
        X_test[col] = X_test[col].fillna('MISSING')
        
        # Fit on train data
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        
        # Transform test data - optimized version
        # Create a mapping dictionary for faster lookup
        class_to_code = {cls: code for code, cls in enumerate(le.classes_)}
        
        # Map test values using the dictionary (faster than apply)
        X_test[col] = X_test[col].map(class_to_code).fillna(-1).astype(int)
        
        label_encoders[col] = le
    
    return X_train, X_test, y_train, label_encoders, low_cardinality_cols, high_cardinality_cols

# Usage example
X, test, y, encoders, low_card_cols, high_card_cols = smart_categorical_encoding_optimized(
    train, test, 'TARGET'
)

model training

In [ ]:

models = []
x = 8
for i in range(x):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.1, random_state=42+i, stratify=y
    )
    
    params_performance = {
        'learning_rate': 0.02,
        'n_estimators': 3000,
        'num_leaves': 255,
        'max_depth': -1,
        'subsample': 0.7,
        'colsample_bytree': 0.6,
        'min_child_samples': 20,
        'reg_alpha': 0.1,
        'reg_lambda': 0.1,
        'n_jobs': -1,
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'random_state': 42+i,
        'feature_fraction': 0.7,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
    }
    
    clf = LGBMClassifier(**params_performance)
    
    clf.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)], 
        early_stopping_rounds=100,
        verbose=50
    )
    models.append(clf)

# Get average predictions from all models
all_predictions = []
for model in models:
    y_pred_proba_single = model.predict_proba(test)[:, 1]
    all_predictions.append(y_pred_proba_single)

# Convert to numpy array for easy averaging
all_predictions = np.array(all_predictions)

# Calculate average prediction across all models
y_pred_proba_avg = np.mean(all_predictions, axis=0)

# Create submission file with average predictions
submission = pd.DataFrame({'SK_ID_CURR': ID_for_test, 'TARGET': y_pred_proba_avg})
submission.to_csv('lgbm_only_numerical_v3.csv', index=False)

In [ ]:
submission.to_csv('submission.csv', index=False)

code for getting feature importance

In [ ]:
# Aggregate feature importance across folds
fi_all = pd.concat(feature_importances, axis=0, ignore_index=True)

# Normalize per fold (optional but helpful when different best_iter across folds)
def _norm_group(df, col='importance_gain'):
    total = df[col].sum()
    if total == 0:
        return df.assign(norm=0.0)
    return df.assign(norm=df[col] / total)

fi_norm = (fi_all.groupby('fold', group_keys=False)
                .apply(_norm_group, col='importance_gain'))

# Average normalized gain across folds
fi_mean = (fi_norm.groupby('feature', as_index=False)['norm']
                .mean()
                .rename(columns={'norm': 'avg_gain_importance'}))

fi_mean = fi_mean.sort_values('avg_gain_importance', ascending=False)

# Plot top-K
TOPK = 40
topk = fi_mean.head(TOPK).iloc[::-1]  # reverse for a nicer horizontal plot

plt.figure(figsize=(8, max(6, TOPK*0.25)))
plt.barh(topk['feature'], topk['avg_gain_importance'])
plt.title(f'Top {TOPK} Features by (Normalized Gain) — Averaged over {N_FOLDS} folds')
plt.xlabel('Average normalized gain importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

# Also keep a CSV for inspection
fi_mean.to_csv('feature_importance_mean_gain.csv', index=False)
print("Saved: feature_importance_mean_gain.csv")